# POD for 0D Isothermal Chemical Reactor
### Importing libraries

In [1]:
import os
import re
import sys
import time
import pyDOE
import itertools
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from PCAfold import PCA, log_transform

sys.path.append(os.path.normpath(os.environ['WORKSPACE_PATH'] + '/ML/PrODE/prode.22.01/'))
from physics.data_generator import DataGenerator

In [2]:
# Set up case directory
path = './RVE/'
if not os.path.exists(path):
    os.makedirs(path)

### Generating Training and Test Data Set
Input parameters

In [3]:
# Training/Testing data
train_cases = 5
test_cases = 1

# Physical model
# =============================================================================
# Input space (limit values)
input_space = {
    'Ttr': [8.e3,   1.5e4],        # Tanslational temperature [K]
    'T0':  [1.e3,   8.e3],         # Initial temperature [K]
    'P0':  [1.e3,   1.e4],         # Initial pressure [Pa]
    'XO2': [0.05e0, 1.e0]          # Initial O2 mole fraction
}

# Time (limit values)
time_limit = [1.e-14, 1.e-1]

# Timescales thermochemistry --------------------------------------------------
groups = ['RVE1', 'RVE3', 'RVE9', 'RVE18']

groups_graph = {
    0: {
        0: {
            0: [0,1],
            1: [2,3],
            2: [4,5],
        },
        1: {
            3: [6,7],
            4: [8,9],
            5: [10,11],
        },
        2: {
            6: [12,13],
            7: [14,15],
            8: [16,17],
        }
    }
}

# Thermochemistry -------------------------------------------------------------
species = {
    'O2': {
        'rovib': {
            'system': 'O3',
            'PES': 'UMN',
            'grouping': 'RVE18'
        }
    },
    'O': None
}

process_flags = {      # Reactive processes
    'excit': 1,
    'diss':  1
}

Solve ODEs

In [4]:
dataGen = DataGenerator(
    path,
    train_cases,
    test_cases,
    time_limit,
    input_space,
    species,
    process_flags,
    groups_graph
)

ic = dataGen.initial_cond()
train_cases = dataGen.generate(ic[0], name='train')
test_cases = dataGen.generate(ic[1], name='test')

  0%|          | 0/10 [00:00<?, ?it/s]

Generating initial conditions ...
Generating train data ...


100%|██████████| 1/1 [00:00<00:00, 1144.11it/s]

Generating test data ...


### Eigenmodes Analysis
Input parameters

In [5]:
split = True     # Split variables
scale = 'linear'     # Transform data
nPC_max = 20      # Max number of preserved principal components

scalings = ['none', 'auto', 'std', 'pareto', 'vast', 'range', '0to1']

Set up PCA case directory

In [6]:
pca_dir = path + 'pca_' + scale
if split:
    pca_dir += '_split'
if not os.path.exists(pca_dir):
    os.makedirs(pca_dir)

Get split feature indeces for separate PCA 

In [7]:
def split_indeces(ls, indeces=[]):
        try:
            sub_ls = []
            start = 0
            for elem in ls:
                sub_ls.append(list(range(start,start+len(elem))))
                start = sub_ls[-1][-1] + 1
            indeces.append(sub_ls)
            ls = list(itertools.chain(*ls))
            return split_indeces(ls, indeces=indeces)
        except:
            return indeces

if split:
    groups_nested_list = dataGen.graph_to_nested_list(groups_graph)
    indeces = [[[0,1]]] + split_indeces(groups_nested_list)

Define error functions

In [8]:
reductions = ['max', 'mean']

def mape(y_true, y_pred, reduction='mean', axis=None):
    err = 100*np.abs((y_true - y_pred) / y_true)
    if reduction =='max':
        red_fn = np.amax
    else:
        red_fn = np.mean
    return red_fn(err, axis=axis)

def mse(y_true, y_pred, reduction='mean', axis=None):
    err = (y_true - y_pred)**2
    if reduction =='max':
        red_fn = np.amax
    else:
        red_fn = np.mean
    return red_fn(err, axis=axis)

Compose the data matrix

In [9]:
acceptance_tol = 1.e-3

if split:
    first = True
    split_fn = lambda y_i, i: [np.take(y_i, j, axis=1) for j in i]
    for case in train_cases:
        y = list(map(split_fn, case[1], indeces))
        if first:
            Y = y
            first = False
        else:
            Y = tf.nest.map_structure(lambda Y_i, y_i: np.hstack((Y_i, y_i)), Y, y)
else:
    first = True
    for case in train_cases:
        y = [[y_i] for y_i in case[1]]
        if first:
            Y = y
            first = False
        else:
            Y = tf.nest.map_structure(lambda Y_i, y_i: np.hstack((Y_i, y_i)), Y, y)
#             for i in range(len(Y)):
#                 for j in range(y[i].shape[1]):
#                     delta = mape(y[i][:,j:j+1], Y[i], axis=0)
#                     if any(delta < acceptance_tol):
#                         continue
#                     else:
#                         Y[i] = np.hstack((Y[i], y[i][:,j:j+1]))

Eigenvalue decomposition

In [10]:
eig_min = 1.e-14   # Minimum eigenvalue to be plotted

eig_dir = pca_dir + '/eigen'
if not os.path.exists(eig_dir):
    os.makedirs(eig_dir)

nPC = {}
for i, y_i in enumerate(Y):
    print(f'>> POD for {groups[i]} coarse-grained model dynamics')
    nPC[groups[i]] = []
    for j, y_ij in enumerate(y_i):
        for s, sc in enumerate(scalings):
            pca = PCA(
                np.log10(y_ij) if scale == 'log' else y_ij,
                scaling=sc,
                n_components=nPC_max,
                use_eigendec=True,
                nocenter=False
            )
            if s == 0:
                L = pca.L.reshape(-1,1)
            else:
                L = np.hstack((L, pca.L.reshape(-1,1)))
        nPC_cg = max(np.argmin(np.abs(L - eig_min), axis=0)) + 1
        nPC[groups[i]].append(nPC_cg)

        fig = plt.figure()
        plt.figure(figsize=(8,7))
        # Y axis
        plt.ylim([eig_min, None])
        plt.ylabel('Eigenvalue')
        plt.yscale('log')
        # X axis
        plt.xlabel('Component Index')
        interval = int(np.ceil(nPC_cg/10))
        plt.xticks(np.arange(0,nPC_cg,interval), labels=np.arange(1,nPC_cg+1,interval))
        plt.xlim([0,nPC_cg-1])
        # Plot
        plt.plot(L, label=scalings)
        plt.legend()
        # Save figure
        fig_path = eig_dir+'/'+groups[i]
        if not os.path.exists(fig_path):
            os.makedirs(fig_path)
        plt.savefig(fig_path+'/split'+str(j+1)+'.pdf')
#         plt.show()
        plt.close()


/home/zanardi/APPLICATIONS/ANACONDA/anaconda3/envs/tf/lib/python3.8/site-packages/PCAfold/reduction.py:225: RuntimeWarning: invalid value encountered in sqrt
  loadings_matrix[i,j] = (self.A[i,j] * np.sqrt(self.L[j])) / np.sqrt(self.S[i,i])
/home/zanardi/APPLICATIONS/ANACONDA/anaconda3/envs/tf/lib/python3.8/site-packages/PCAfold/reduction.py:234: RuntimeWarning: invalid value encountered in sqrt
  tqj[i,j] = ( (self.A[i,j] * np.sqrt(self.L[j])) / (np.sqrt(self.S[i,i])) )**2 / constant_factor
findfont: Font family ['serif'] not found. Falling back to DejaVu Sans.
findfont: Generic family 'serif' not found because none of the following families were found: Arial


>> POD for RVE1 coarse-grained model dynamics


findfont: Font family ['cursive'] not found. Falling back to DejaVu Sans.
findfont: Generic family 'cursive' not found because none of the following families were found: Arial
findfont: Font family ['Charter'] not found. Falling back to DejaVu Sans.
findfont: Font family ['monospace'] not found. Falling back to DejaVu Sans.
findfont: Generic family 'monospace' not found because none of the following families were found: Arial
findfont: Font family ['Charter'] not found. Falling back to DejaVu Sans.
findfont: Font family ['Charter'] not found. Falling back to DejaVu Sans.
findfont: Font family ['sans'] not found. Falling back to DejaVu Sans.
findfont: Generic family 'sans' not found because none of the following families were found: Arial


>> POD for RVE3 coarse-grained model dynamics
>> POD for RVE9 coarse-grained model dynamics
>> POD for RVE18 coarse-grained model dynamics


/home/zanardi/APPLICATIONS/ANACONDA/anaconda3/envs/tf/lib/python3.8/site-packages/PCAfold/reduction.py:225: RuntimeWarning: invalid value encountered in sqrt
  loadings_matrix[i,j] = (self.A[i,j] * np.sqrt(self.L[j])) / np.sqrt(self.S[i,i])
/home/zanardi/APPLICATIONS/ANACONDA/anaconda3/envs/tf/lib/python3.8/site-packages/PCAfold/reduction.py:234: RuntimeWarning: invalid value encountered in sqrt
  tqj[i,j] = ( (self.A[i,j] * np.sqrt(self.L[j])) / (np.sqrt(self.S[i,i])) )**2 / constant_factor


...

<Figure size 432x288 with 0 Axes>

<Figure size 432x288 with 0 Axes>

<Figure size 432x288 with 0 Axes>

<Figure size 432x288 with 0 Axes>

<Figure size 432x288 with 0 Axes>

Reconstruct variables and evaluate error

In [11]:
err_dir = pca_dir + '/err'
if not os.path.exists(err_dir):
    os.makedirs(err_dir)
    
for i, y_i in enumerate(Y):
    print(f'>> Reconstructed PCA error for {groups[i]} coarse-grained model dynamics')
    for j, y_ij in enumerate(y_i):
        nPC_cg = nPC[groups[i]][j]
        # Initilize error arrays
        shape = (nPC_cg, len(scalings))
        mape_i = {r: np.zeros(shape) for r in reductions}
        mse_i = {r: np.zeros(shape) for r in reductions}
        # Reconstruct variables for an increasing number of principal components
        for k, nPC_k in enumerate(range(1, nPC_cg+1)):
            # Loop over the different type of scalings
            for s, sc in enumerate(scalings):
                # Initialize the PCA object
                pca = PCA(
                    np.log10(y_ij) if scale == 'log' else y_ij,
                    scaling=sc,
                    n_components=nPC_k,
                    use_eigendec=True,
                    nocenter=False
                )
                # Calculate the principal components
                pc = pca.transform(np.log10(y_ij) if scale == 'log' else y_ij)
                # Calculate the reconstructed variables
                y_rec = pca.reconstruct(pc)
                y_rec = np.power(10,y_rec) if scale == 'log' else y_rec
                # Calculate errors
                for r in reductions:
                    mape_i[r][k,s] = mape(y_ij, y_rec, reduction=r)
                    mse_i[r][k,s] = mse(y_ij, y_rec, reduction=r)

        PC_mape_i = {r: max(np.argmin(np.abs(mape_i[r] - 1.e-1), axis=0))+1 for r in reductions}
        PC_mse_i = {r: max(np.argmin(np.abs(mse_i[r] - 1.e-1), axis=0))+1 for r in reductions}

        # Plot errors
        n, m = 2, 2
        fig, axs = plt.subplots(n, m, sharex=True, figsize=(12,10))
        for n_ in range(n):
            for m_ in range(m):
                if n_ == n-1:
                    err, PC = mape_i, PC_mape_i
                    axs[n_,0].set_ylabel('MAPE')
                else:
                    err, PC = mse_i, PC_mse_i
                    axs[n_,0].set_ylabel('MSE')
                # Text
                axs[n_,m_].text(
                    0.5, 0.05,
                    f'PC>{PC[reductions[m_]]} for err<0.1',
                    horizontalalignment='center',
                    verticalalignment='center',
                    transform=axs[n_,m_].transAxes
                )
                # X axis
                axs[n_,m_].set_xlim([0,nPC_cg-1])
                interval = int(np.ceil(nPC_cg/10))
                axs[n_,m_].set_xticks(np.arange(0,nPC_cg,interval))
                axs[n_,m_].set_xticklabels(np.arange(1,nPC_cg+1,interval))
                # Y axis
                axs[n_,m_].set_yscale('log')
                if n_ == n-1:
                    axs[n_,m_].set_xlabel('# of Preserved Components')
                # Plot
                axs[n_,m_].plot(err[reductions[m_]], label=scalings)
                axs[n_,m_].legend(fontsize=12)
        axs[0,0].set_title('Max')
        axs[0,1].set_title('Mean')
        plt.tight_layout()
        # Save figure
        fig_path = err_dir+'/'+groups[i]
        if not os.path.exists(fig_path):
            os.makedirs(fig_path)
        plt.savefig(fig_path+'/split'+str(j+1)+'.pdf')
#         plt.show()
        plt.close()


>> Reconstructed PCA error for RVE1 coarse-grained model dynamics


findfont: Font family ['serif'] not found. Falling back to DejaVu Sans.
findfont: Generic family 'serif' not found because none of the following families were found: Arial
findfont: Font family ['serif'] not found. Falling back to DejaVu Sans.
findfont: Generic family 'serif' not found because none of the following families were found: Arial


>> Reconstructed PCA error for RVE3 coarse-grained model dynamics
>> Reconstructed PCA error for RVE9 coarse-grained model dynamics
>> Reconstructed PCA error for RVE18 coarse-grained model dynamics


/home/zanardi/APPLICATIONS/ANACONDA/anaconda3/envs/tf/lib/python3.8/site-packages/PCAfold/reduction.py:225: RuntimeWarning: invalid value encountered in sqrt
  loadings_matrix[i,j] = (self.A[i,j] * np.sqrt(self.L[j])) / np.sqrt(self.S[i,i])
/home/zanardi/APPLICATIONS/ANACONDA/anaconda3/envs/tf/lib/python3.8/site-packages/PCAfold/reduction.py:234: RuntimeWarning: invalid value encountered in sqrt
  tqj[i,j] = ( (self.A[i,j] * np.sqrt(self.L[j])) / (np.sqrt(self.S[i,i])) )**2 / constant_factor
